# Simple Turbulence Tests

## Imports

In [ ]:
# IMPORTANT: check gpustat --watch before 
# running scripts on the cluster
# BETTER NOT USE NOTEBOOKS AT ALL,
# THEY BLOCK THE GPU MEMORY IF NOT
# RESET PROPERLY
import os
# set the correct GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
# you may also use
# # ==== GPU selection ====
# from autocvd import autocvd
# autocvd(num_gpus = 1)
# # =======================
# in regular python scripts

# numerics
import jax
import jax.numpy as jnp
import numpy as np
# timing
from timeit import default_timer as timer

# plotting
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# jf1uids.jf1uids data structures
from jf1uids import SimulationConfig
from jf1uids import SimulationParams
from jf1uids.option_classes import WindConfig
from jf1uids.option_classes.simulation_config import BACKWARDS, OSHER, FORWARDS

# jf1uids.jf1uids setup functions
from jf1uids import get_helper_data
from jf1uids.fluid_equations.fluid import construct_primitive_state
from jf1uids import get_registered_variables
from jf1uids.option_classes.simulation_config import finalize_config

# turbulent ic setup
from jf1uids.initial_condition_generation.turb import create_turb_field

# main simulation function
from jf1uids import time_integration

# units
from jf1uids import CodeUnits
from astropy import units as u
import astropy.constants as c

import random

## Initiating the stellar wind simulation

In [ ]:
def setup_turbulent_sim(t_end = None, snapshots = 10):
    print("👷 Setting up simulation...")

    # simulation settings
    gamma = 5/3

    # spatial domain
    box_size = 1.0

    # resolution
    num_cells = 128

    # turbulence
    wanted_rms = 50 * u.km / u.s

    fixed_timestep = True
    dt_max = 0.1

    mhd = False
    # setup simulation config
    config = SimulationConfig(
        runtime_debugging = False,
        first_order_fallback = False,
        progress_bar = False,
        dimensionality = 3,
        num_ghost_cells = 2,
        box_size = box_size, 
        num_cells = num_cells,
        mhd = mhd,
        fixed_timestep = fixed_timestep,
        differentiation_mode = FORWARDS,
        return_snapshots = True,
        num_snapshots = snapshots,
    )

    helper_data = get_helper_data(config)
    registered_variables = get_registered_variables(config)

    # setup the unit system
    code_length = 3 * u.parsec
    code_mass = 1 * u.M_sun
    code_velocity = 100 * u.km / u.s
    code_units = CodeUnits(code_length, code_mass, code_velocity)

    # time domain
    C_CFL = 0.4

    # set the final time of the simulation
    if t_end is None:
        t_final = 1.0 * 1e4 * u.yr
        t_end = t_final.to(code_units.code_time).value

    # set the simulation parameters
    params = SimulationParams(
        C_cfl=C_CFL,
        dt_max=dt_max,
        gamma=gamma,
        t_end=t_end,
    )

    # homogeneous initial state
    rho_0 = 2 * c.m_p / u.cm**3
    p_0 = 3e4 * u.K / u.cm**3 * c.k_B

    rho = (
        jnp.ones((config.num_cells, config.num_cells, config.num_cells))
        * rho_0.to(code_units.code_density).value
    )

    # turbulence parameters
    turbulence_slope = -2
    kmin = 2
    kmax = 64
    p = (
        jnp.ones((config.num_cells, config.num_cells, config.num_cells))
        * p_0.to(code_units.code_pressure).value
    )

    u_x = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax, 0)
    u_y = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax, 0)
    u_z = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax, 0)

    # scale the turbulence to the desired rms velocity
    rms_vel = jnp.sqrt(jnp.mean(u_x**2 + u_y**2 + u_z**2))

    u_x = u_x / rms_vel * wanted_rms.to(code_units.code_velocity).value
    u_y = u_y / rms_vel * wanted_rms.to(code_units.code_velocity).value
    u_z = u_z / rms_vel * wanted_rms.to(code_units.code_velocity).value

    initial_state = construct_primitive_state(
        config=config,
        registered_variables=registered_variables,
        density=rho,
        velocity_x=u_x,
        velocity_y=u_y,
        velocity_z=u_z,
        gas_pressure=p,
    )

    config = finalize_config(config, initial_state.shape)
    return initial_state, config, params, helper_data, registered_variables


## Setting the simulation parameters and initial state

In [ ]:
initial_state, config, params, helper_data, registered_variables = setup_turbulent_sim()
result = time_integration(initial_state, config, params, helper_data, registered_variables)


In [ ]:
np.shape(result.states)

In [ ]:
def turb_sim_n_forward(initial_state, end_time, n_snapshots):
    print("👷 Setting up simulation...")

    gamma = 5 / 3
    box_size = 1.0
    num_cells = 128
    fixed_timestep = True
    dt_max = 0.1
    mhd = False

    # setup simulation config
    config = SimulationConfig(
        runtime_debugging=False,
        first_order_fallback=False,
        progress_bar=False,
        dimensionality=3,
        num_ghost_cells=2,
        box_size=box_size,
        num_cells=num_cells,
        mhd=mhd,
        fixed_timestep=fixed_timestep,
        differentiation_mode=FORWARDS,
        return_snapshots=True,
        num_snapshots=n_snapshots,
    )

    helper_data = get_helper_data(config)
    registered_variables = get_registered_variables(config)

    # setup the unit system
    code_length = 3 * u.parsec
    code_mass = 1 * u.M_sun
    code_velocity = 100 * u.km / u.s
    code_units = CodeUnits(code_length, code_mass, code_velocity)

    # time domain
    C_CFL = 0.4

    # set the final time of the simulation

    # set the simulation parameters
    params = SimulationParams(
        C_cfl=C_CFL,
        dt_max=dt_max,
        gamma=gamma,
        t_end=end_time,
    )

    initial_state = construct_primitive_state(
        config=config,
        registered_variables=registered_variables,
        density=initial_state[0],
        velocity_x=initial_state[1],
        velocity_y=initial_state[2],
        velocity_z=initial_state[3],
        gas_pressure=initial_state[4],
    )

    config = finalize_config(config, initial_state.shape)
    return config, params, helper_data, registered_variables


In [ ]:
code_length = 3 * u.parsec
code_mass = 1 * u.M_sun
code_velocity = 100 * u.km / u.s
code_units = CodeUnits(code_length, code_mass, code_velocity)
states = []
t_final = 1.0 * 1e4
max_snapshots = 80
snapshots_epoch = 10
epochs = max_snapshots // snapshots_epoch


for i in range (epochs):
    print("epoch", i)
    t_end_epoch = (t_final / epochs) * (i+1)
    t_end_epoch = (t_end_epoch * u.yr).to(code_units.code_time).value
    if i == 0:
        initial_state, config, params, helper_data, registered_variables = setup_turbulent_sim(t_end_epoch)
    else:
        initial_state = states[-1]
        config, params, helper_data, registered_variables = turb_sim_n_forward(initial_state, t_end_epoch, snapshots_epoch)

    result = time_integration(
        initial_state, config, params, helper_data, registered_variables
    )
    states.extend(result.states)

In [ ]:
run_on_batches_states = states

In [ ]:
initial_state, config, params, helper_data, registered_variables = setup_turbulent_sim(snapshots = 80)
result = time_integration(
    initial_state, config, params, helper_data, registered_variables
)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

run_at_once_states = result.states
run_on_batches_states = np.array(run_on_batches_states)
run_at_once_states = np.array(run_at_once_states)
# Parameters
z_level = 128 // 2
slice_indices = [i * 10 for i in range(8)]  # 0, 10, ..., 90

# Create figure with 10 rows and 2 columns
fig, axes = plt.subplots(nrows=8, ncols=2, figsize=(8, 40))
fig.subplots_adjust(hspace=0.3)

for i, idx in enumerate(slice_indices):
    # Left column: run_on_batches
    ax_left = axes[i, 0]
    im1 = ax_left.imshow(
        run_on_batches_states[idx, 0, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        norm=LogNorm(),
    )
    ax_left.set_title(f"Batch {idx} (Batched)")
    ax_left.set_ylabel(f"Batch {idx}", fontsize=10)
    ax_left.set_xticks([])
    ax_left.set_yticks([])

    # Right column: run_at_once
    ax_right = axes[i, 1]
    im2 = ax_right.imshow(
        run_at_once_states[idx, 0, :, :, z_level].T,
        origin="lower",
        extent=[0, 1, 0, 1],
        norm=LogNorm(),
    )
    ax_right.set_title(f"Batch {idx} (All-at-once)")
    ax_right.set_xticks([])
    ax_right.set_yticks([])

# Set common labels for columns
axes[0, 0].set_xlabel("X")
axes[0, 1].set_xlabel("X")

# Optionally add colorbars if needed
# fig.colorbar(im1, ax=axes[:, 0], orientation='vertical')
# fig.colorbar(im2, ax=axes[:, 1], orientation='vertical')

plt.tight_layout()
plt.show()


In [ ]:
def check_array(name, arr):
    if not jnp.all(jnp.isfinite(arr)):
        print(f"🚨 {name} contains NaN or Inf!")
        raise ValueError(f"{name} invalid")

In [ ]:
a = num_cells // 2 - 10
b = num_cells // 2 + 10
p = jnp.ones((config.num_cells, config.num_cells, config.num_cells)) * p_0.to(code_units.code_pressure).value

u_x = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax)
u_y = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax)
u_z = create_turb_field(config.num_cells, 1, turbulence_slope, kmin, kmax)

# scale the turbulence to the desired rms velocity
rms_vel = jnp.sqrt(jnp.mean(u_x**2 + u_y**2 + u_z**2))

u_x = u_x / rms_vel * wanted_rms.to(code_units.code_velocity).value
u_y = u_y / rms_vel * wanted_rms.to(code_units.code_velocity).value
u_z = u_z / rms_vel * wanted_rms.to(code_units.code_velocity).value

if mhd:
    grid_spacing = config.box_size / config.num_cells
    x = jnp.linspace(
        grid_spacing / 2, config.box_size - grid_spacing / 2, config.num_cells
    )
    y = jnp.linspace(
        grid_spacing / 2, config.box_size - grid_spacing / 2, config.num_cells
    )
    X, Y = jnp.meshgrid(x, y, indexing="ij")

    B_0 = 1 / jnp.sqrt(2)
    B_x = B_0 * jnp.ones_like(X)
    B_y = B_0 * jnp.ones_like(X)
    B_z = jnp.zeros_like(X)
    initial_state = construct_primitive_state(
        config=config,
        registered_variables=registered_variables,
        density=rho,
        velocity_x=u_x,
        velocity_y=u_y,
        velocity_z=u_z,
        gas_pressure=p,
        magnetic_field_x=B_x,
        magnetic_field_y=B_y,
        magnetic_field_z=B_z,
    )
else:
    initial_state = construct_primitive_state(
        config = config,
        registered_variables=registered_variables,
        density = rho,
        velocity_x = u_x,
        velocity_y = u_y,
        velocity_z = u_z,
        gas_pressure = p
    )

config = finalize_config(config, initial_state.shape)
result = time_integration(initial_state, config, params, helper_data, registered_variables)
final_state = result.states[-1]


In [ ]:
print(result.total_energy[0])

## Simulation and Gradient

In [ ]:
from jf1uids.fluid_equations.fluid import get_absolute_velocity

def internal_energy_spectrum(state, gamma, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells
    p = state[registered_variables.pressure_index]

    if config.cosmic_ray_config.cosmic_rays:
        gamma_cr = 4/3
        p = p - state[registered_variables.cosmic_ray_n_index] ** gamma_cr

    internal_energy = p / (gamma - 1)

    # Remove ghost cells if necessary
    if config.dimensionality == 1:
        internal_energy = internal_energy[num_ghost_cells:-num_ghost_cells]
    else:
        slices = tuple(slice(num_ghost_cells, -num_ghost_cells) for _ in range(config.dimensionality))
        internal_energy = internal_energy[slices]

    internal_energy_np = np.array(internal_energy)

    fft_energy = np.fft.fftn(internal_energy_np)
    fft_energy_shifted = np.fft.fftshift(fft_energy)
    power_spectrum = np.abs(fft_energy_shifted)**2

    return power_spectrum

def calculate_kinetic_energy(state, helper_data, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells

    rho = state[registered_variables.density_index]
    u = get_absolute_velocity(state, config, registered_variables)

    kinetic_energy = 0.5 * rho * u ** 2

    if config.dimensionality == 1:
        return jnp.sum(kinetic_energy[num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells])
    else:
        return jnp.sum(kinetic_energy * config.grid_spacing**config.dimensionality)

def kinetic_energy_spectrum(state, helper_data, config, registered_variables):
    num_ghost_cells = config.num_ghost_cells
    rho = state[registered_variables.density_index]


    u = get_absolute_velocity(state, config, registered_variables)

    kinetic_energy = 0.5 * rho * u ** 2

    if config.dimensionality == 1:
        kinetic_energy =  kinetic_energy[num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells]
    else:
         kinetic_energy =  kinetic_energy * config.grid_spacing**config.dimensionality

    kinetic_energy_np = np.array(kinetic_energy)

    fft_energy = np.fft.fftn(kinetic_energy_np)
    fft_energy_shifted = np.fft.fftshift(fft_energy)
    power_spectrum = np.abs(fft_energy_shifted)**2

    return power_spectrum

def mass_spectrum(
    state,
    helper_data,
    config,
):
    num_ghost_cells = config.num_ghost_cells

    if config.dimensionality == 1:
        mass = state[0, num_ghost_cells:-num_ghost_cells] * helper_data.cell_volumes[num_ghost_cells:-num_ghost_cells]
    else:
        slice_off_ghost_cells = (0,) + (slice(num_ghost_cells, -num_ghost_cells),) * config.dimensionality
        # note that here the box size is assumed to be the box size without the ghost cells
        mass = state[slice_off_ghost_cells] * config.box_size**config.dimensionality
    
    mass_np = np.array(mass)

    fft_mass = np.fft.fftn(mass_np)
    fft_mass_shifted = np.fft.fftshift(fft_mass)
    power_spectrum = np.abs(fft_mass_shifted)**2

    return power_spectrum

def radial_spectrum(power_spectrum):
    shape = power_spectrum.shape
    center = [s // 2 for s in shape]
    
    # Create coordinate grids
    z, y, x = np.indices(shape)
    k = np.sqrt((x - center[2])**2 + (y - center[1])**2 + (z - center[0])**2)
    k = k.astype(int)

    # Bin average
    k_max = int(np.max(k))
    spectrum = np.zeros(k_max + 1)
    counts = np.zeros(k_max + 1)
    
    for i in range(k_max + 1):
        mask = (k == i)
        spectrum[i] = power_spectrum[mask].sum()
        counts[i] = mask.sum()

    return spectrum / np.maximum(counts, 1)



In [ ]:
internal_e_spectrum = radial_spectrum(internal_energy_spectrum(    
    state = result.states[0], 
    gamma = gamma,
    config = config,
    registered_variables = registered_variables))

kinetic_e_spectrum = radial_spectrum(kinetic_energy_spectrum(    
    state=result.states[0], 
    helper_data=helper_data,
    config=config,
    registered_variables=registered_variables))

mass_spectrum = radial_spectrum(mass_spectrum(    
    state=result.states[0], 
    helper_data=helper_data,
    config=config))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (15, 5))
ax[0].scatter(range(0, len(internal_e_spectrum)), internal_e_spectrum)
ax[0].set_yscale('log')
ax[0].set_title('Internal energy')

ax[1].scatter(range(0, len(kinetic_e_spectrum)), kinetic_e_spectrum)
ax[1].set_yscale('log')
ax[1].set_xscale('log')
ax[1].set_title('Kinetic energy')

ax[2].scatter(range(0, len(mass_spectrum)), mass_spectrum)
ax[2].set_yscale('log')
ax[2].set_title('mass spectrum')

#ax[2].scatter(range(0, len(kinetic_e_spectrum)), internal_e_spectrum + kinetic_e_spectrum)
#ax[2].set_yscale('log')
#ax[2].set_title('kin + int spectrum')

## Visualization

### Cut through the Simulation

In [ ]:
from matplotlib.colors import LogNorm

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# plot cuts at this z level
z_level = num_cells // 2

ax1.imshow(final_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
ax1.set_title("Density")

ax2.imshow(jnp.sqrt(final_state[1, :, :, z_level]**2 + final_state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
ax2.set_title("Velocity")

ax3.imshow(final_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
ax3.set_title("Pressure")

# equal aspect ratio
ax1.set_aspect('equal', 'box')
ax2.set_aspect('equal', 'box')
ax3.set_aspect('equal', 'box')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


In [ ]:
# Assuming result and num_cells are defined elsewhere
states = result.states
z_level = random.randint(0, num_cells - 1)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Scalar field (e.g., density)
cax0 = axs[0].imshow(
    states[0, 0, :, :, z_level].T,
    origin="lower",
    norm=plt.Normalize(vmin=0, vmax=1),
)
fig.colorbar(cax0, ax=axs[0])
axs[0].set_title("Density")
axs[0].set_xlabel("x")
axs[0].set_ylabel("y")

# Plot 2: Vector magnitude (e.g., velocity magnitude)
cax1 = axs[1].imshow(
    jnp.sqrt(
        states[0, 1, :, :, z_level] ** 2
        + states[0, 2, :, :, z_level] ** 2
        + states[0, 3, :, :, z_level] ** 2
    ).T,
    origin="lower",
    norm=plt.Normalize(vmin=0, vmax=1),
)
fig.colorbar(cax1, ax=axs[1])
axs[1].set_title("Velocity Magnitude")
axs[1].set_xlabel("x")
axs[1].set_ylabel("y")

# Plot 3: Optional third field, e.g., pressure or similar (reusing animate_vector logic)
cax2 = axs[2].imshow(
    jnp.sqrt(
        states[0, 1, :, :, z_level] ** 2
        + states[0, 2, :, :, z_level] ** 2
        + states[0, 3, :, :, z_level] ** 2
    ).T,
    origin="lower",
    norm=plt.Normalize(vmin=0, vmax=1),
)
fig.colorbar(cax2, ax=axs[2])
axs[2].set_title("Pressure")
axs[2].set_xlabel("x")
axs[2].set_ylabel("y")


# Update function for all three plots
def animate_all(i):
    cax0.set_array(states[i, 0, :, :, z_level].T)
    cax1.set_array(
        jnp.sqrt(
            states[i, 1, :, :, z_level] ** 2
            + states[i, 2, :, :, z_level] ** 2
            + states[i, 3, :, :, z_level] ** 2
        ).T
    )
    cax2.set_array(states[i, 4, :, :, z_level].T)
    return cax0, cax1, cax2


ani = animation.FuncAnimation(fig, animate_all, frames=states.shape[0], interval=50)

ani.save("turbulence_sr/figures/turb_all.gif")
plt.show()


In [ ]:
states = result.states
z_level = random.randint(0, num_cells - 1)

data = states[20, 0, :, :, :].T

data_shape = states.shape[2:]

# Create coordinate grid
x, y, z = np.indices(data_shape)
edge_mask = (
    (x == 0)
    | (x == data_shape[0] - 1)
    | (y == 0)
    | (y == data_shape[1] - 1)
    | (z == 0)
    | (z == data_shape[2] - 1)
)

x_edge = x[edge_mask]
y_edge = y[edge_mask]
z_edge = z[edge_mask]

# Normalize for consistent colormap
vmin = np.min(states)
vmax = np.max(states)
norm = plt.Normalize(vmin, vmax)
cmap = cm.plasma

# Initial color values
initial_vals = states[0, 0, :, :, :].T[edge_mask]
initial_colors = cmap(norm(initial_vals))

# Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(x_edge, y_edge, z_edge, c=initial_colors, marker="s", s=20, alpha=0.9)

plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="Pressure")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D Pressure Animation (Edge Voxels)")


# Animation update function
def animate_all(i):
    frame_vals = states[i, 0, :, :, :].T[edge_mask]
    frame_colors = cmap(norm(frame_vals))
    sc.set_facecolor(frame_colors)
    return (sc,)


# Animate
ani = animation.FuncAnimation(
    fig, animate_all, frames=states.shape[0], interval=50, blit=False
)

ani.save("turbulence_sr/figures/3d_pressure.gif", writer="pillow")
plt.show()